# Import Libraries

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load and Analyze Dataset

In [16]:
housing = pd.read_csv('housing.csv')
print("Shape:", housing.shape)
print("\nColumn dtypes:\n", housing.dtypes)
print("\nMissing values:\n", housing.isnull().sum())
print("\nTarget stats:\n", housing['median_house_value'].describe())

Shape: (20640, 10)

Column dtypes:
 longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object

Missing values:
 longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

Target stats:
 count     20640.000000
mean     206855.816909
std      115395.615874
min       14999.000000
25%      119600.000000
50%      179700.000000
75%      264725.000000
max      500001.000000
Name: median_house_value, dtype: float64


In [17]:
X = housing.drop('median_house_value',axis=1)
y = housing['median_house_value']

In [18]:
numeric_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                     'total_bedrooms', 'population', 'households', 'median_income']
categorical_features = ['ocean_proximity']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


Train size: 16512, Test size: 4128


In [21]:
#Numeric: median impute (for total_bedrooms) + StandardScaler
# Categorical: One-Hot Encode (ocean_proximity has 5 categories, no order -> OHE not ordinal)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)])

In [22]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(random_state=42),
    'Lasso Regression': Lasso(random_state=42, max_iter=5000),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),}

In [23]:
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append({'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2})
    print(f"{name:20s} | MAE: {mae:10.2f} | RMSE: {rmse:10.2f} | R2: {r2:.4f}")

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print("\n===== MODEL COMPARISON (sorted by R2) =====")
print(results_df)

Linear Regression    | MAE:   50670.49 | RMSE:   70059.19 | R2: 0.6254
Ridge Regression     | MAE:   50676.92 | RMSE:   70066.02 | R2: 0.6254
Lasso Regression     | MAE:   50671.48 | RMSE:   70060.11 | R2: 0.6254
Decision Tree        | MAE:   43604.01 | RMSE:   69175.77 | R2: 0.6348
Random Forest        | MAE:   31628.41 | RMSE:   48941.70 | R2: 0.8172
Gradient Boosting    | MAE:   38278.15 | RMSE:   55903.12 | R2: 0.7615

===== MODEL COMPARISON (sorted by R2) =====
               Model           MAE           MSE          RMSE        R2
0      Random Forest  31628.407311  2.395290e+09  48941.700343  0.817210
1  Gradient Boosting  38278.148174  3.125159e+09  55903.124191  0.761513
2      Decision Tree  43604.014293  4.785287e+09  69175.769189  0.634825
3  Linear Regression  50670.489236  4.908291e+09  70059.193339  0.625438
4   Lasso Regression  50671.484863  4.908418e+09  70060.105989  0.625429
5   Ridge Regression  50676.922171  4.909247e+09  70066.021121  0.625365


In [24]:
best_model_name = results_df.iloc[0]['Model']
print(f"\nBest baseline model: {best_model_name}")

# Random Forest and Gradient Boosting typically top this dataset.
# We'll tune Random Forest with GridSearchCV.
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 20],
    'model__min_samples_leaf': [1, 2]}

grid_search = GridSearchCV(
    rf_pipe, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("\nBest Params:", grid_search.best_params_)
print("Best CV R2:", grid_search.best_score_)

best_rf = grid_search.best_estimator_
tuned_preds = best_rf.predict(X_test)

tuned_mae = mean_absolute_error(y_test, tuned_preds)
tuned_mse = mean_squared_error(y_test, tuned_preds)
tuned_rmse = np.sqrt(tuned_mse)
tuned_r2 = r2_score(y_test, tuned_preds)

print(f"\n===== TUNED RANDOM FOREST (Test Set) =====")
print(f"MAE:  {tuned_mae:.2f}")
print(f"MSE:  {tuned_mse:.2f}")
print(f"RMSE: {tuned_rmse:.2f}")
print(f"R2:   {tuned_r2:.4f}")


Best baseline model: Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best Params: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Best CV R2: 0.8132938432417826

===== TUNED RANDOM FOREST (Test Set) =====
MAE:  31450.15
MSE:  2380761257.22
RMSE: 48793.05
R2:   0.8183


In [25]:
summary = {
    'best_baseline_model': best_model_name,
    'baseline_r2': results_df.iloc[0]['R2'],
    'tuned_model': 'Random Forest (Tuned)',
    'tuned_mae': tuned_mae,
    'tuned_mse': tuned_mse,
    'tuned_rmse': tuned_rmse,
    'tuned_r2': tuned_r2,
    'best_params': grid_search.best_params_}

In [26]:
summary


{'best_baseline_model': 'Random Forest',
 'baseline_r2': np.float64(0.8172104989933294),
 'tuned_model': 'Random Forest (Tuned)',
 'tuned_mae': 31450.153822069635,
 'tuned_mse': 2380761257.2234473,
 'tuned_rmse': np.float64(48793.04517268263),
 'tuned_r2': 0.8183192196668712,
 'best_params': {'model__max_depth': None,
  'model__min_samples_leaf': 2,
  'model__n_estimators': 200}}